<a href="https://colab.research.google.com/github/RautRitesh/langgraph/blob/main/langchain_agentic_chunking.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Using llm for doing chunking of document

In [ ]:
pip install langchain langchain-community langchain-huggingface langchain-groq pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 86.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 5.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google

In [ ]:
from langchain_core.output_parsers import JsonOutputToolsParser
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_classic.chains import create_extraction_chain, create_extraction_chain_pydantic
from typing import Optional, List
from pydantic import BaseModel
from langchain_classic import hub
from langchain_community.docstore.document import Document
import pandas as pd

In [ ]:
df=pd.read_csv("structured_cases_v3.csv")
paragraphs=df['AGENT_CHUNK'].tolist()

In [ ]:
from google.colab import userdata
api_key=userdata.get('groq_api_key_2')

In [ ]:
obj=hub.pull("wfh/proposal-indexing")
print(obj)
llm=ChatGroq(model="llama-3.3-70b-versatile",api_key=api_key)
runnable=obj|llm

/tmp/ipykernel_7974/1130794405.py:1: LangChainDeprecationWarning: langchain_classic.hub.pull is deprecated. Use the LangSmith SDK instead.
  obj=hub.pull("wfh/proposal-indexing")


input_variables=['input'] input_types={} partial_variables={} metadata={'lc_hub_owner': 'wfh', 'lc_hub_repo': 'proposal-indexing', 'lc_hub_commit_hash': 'd962e1728e4cb8a6c7f0aa05522c4102fc8c941c20a6915ff3b7f243cca93943'} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='Decompose the "Content" into clear and simple propositions, ensuring they are interpretable out of\ncontext.\n1. Split compound sentence into simple sentences. Maintain the original phrasing from the input\nwhenever possible.\n2. For any named entity that is accompanied by additional descriptive information, separate this\ninformation into its own distinct proposition.\n3. Decontextualize the proposition by adding necessary modifier to nouns or entire sentences\nand replacing pronouns (e.g., "it", "he", "she", "they", "this", "that") with the full name of the\nentities they refer to.\n4. Present the results as a list of strings, formatted in J

In [ ]:
class Sentences(BaseModel):
  sentences:List[str]


extraction_chain=llm.with_structured_output(Sentences)


In [ ]:
result=extraction_chain.invoke("Hi hello who are you")

In [ ]:
print(result)

sentences=["I'm an AI assistant, and I'm here to help answer your questions.", 'How can I assist you today?']


In [ ]:
def get_propositions(text):
  runnable_output=runnable.invoke({
      "input":text
  }).content
  propositions=extraction_chain.invoke(runnable_output)
  propositions=propositions.sentences
  return propositions




In [ ]:
eassy_paragraphs=[]
i=0
for para in paragraphs:
  propositions=get_propositions(para)
  eassy_paragraphs.extend(propositions)
  print(f"Done {i+1}")
  i+1


##New updated code for agentic chunker and group

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
import uuid
from langchain.chat_models import ChatGroq
import os
from typing import Optional
from pydantic import BaseModel
from langchain_classic.chains import create_extraction_chain_pydantic

class AgenticChunker:
    def __init__(self, api_key=None):
        self.chunks = {}
        self.id_truncate_limit = 5

        # Whether or not to update/refine summaries and titles as you get new information
        self.generate_new_metadata_ind = True
        self.print_logging = True

        if api_key is None:
            openai_api_key = api_key

        if api_key is None:
            raise ValueError("API key is not provided and not found in environment variables")

        self.llm = ChatGroq(model='llama-3.3-70b-versatile', api_key=api_key, temperature=0)

    def add_propositions(self, propositions):
        for proposition in propositions:
            self.add_proposition(proposition)

    def add_proposition(self, proposition):
        if self.print_logging:
            print (f"\nAdding: '{proposition}'")

        # If it's your first chunk, just make a new chunk and don't check for others
        if len(self.chunks) == 0:
            if self.print_logging:
                print ("No chunks, creating a new one")
            self._create_new_chunk(proposition)
            return

        chunk_id = self._find_relevant_chunk(proposition)

        # If a chunk was found then add the proposition to it
        if chunk_id:
            if self.print_logging:
                print (f"Chunk Found ({self.chunks[chunk_id]['chunk_id']}), adding to: {self.chunks[chunk_id]['title']}")
            self.add_proposition_to_chunk(chunk_id, proposition)
            return
        else:
            if self.print_logging:
                print ("No chunks found")
            # If a chunk wasn't found, then create a new one
            self._create_new_chunk(proposition)


    def add_proposition_to_chunk(self, chunk_id, proposition):
        # Add then
        self.chunks[chunk_id]['propositions'].append(proposition)

        # Then grab a new summary
        if self.generate_new_metadata_ind:
            self.chunks[chunk_id]['summary'] = self._update_chunk_summary(self.chunks[chunk_id])
            self.chunks[chunk_id]['title'] = self._update_chunk_title(self.chunks[chunk_id])

    def _update_chunk_summary(self, chunk):
        """
        If you add a new proposition to a chunk, you may want to update the summary or else they could get stale
        """
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a Legal Documentation Expert managing clusters of sentences (chunks) extracted from Nepali Supreme Court judgments.
                    A new proposition was just added to your chunk. Generate a precise, 1-sentence legal summary that defines the exact scope of this chunk.

                    A good summary will clarify the specific legal sub-topic this chunk covers (e.g., 'Arguments regarding the partition of private property' or 'The court's application of the Contract Act due to license revocation').

                    Anticipate legal generalization: If the chunk contains specific plot numbers (Kitta No. 917) or monetary amounts, generalize it to "disputed property" or "pledged collateral amount" while maintaining the legal consequence.

                    Example:
                    Input: Proposition: The plaintiff was only liable for Rs. 1,00,000 as a guarantor.
                    Output: This chunk details the legal boundaries of a guarantor's financial liability under the loan agreement.

                    Only respond with the chunk's new summary, nothing else.
                    """,
                ),
                ("user", "Chunk's propositions:\n{proposition}\n\nCurrent chunk summary:\n{current_summary}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_summary = runnable.invoke({
            "proposition": "\n".join(chunk['propositions']),
            "current_summary" : chunk['summary']
        }).content

        return new_chunk_summary

    def _update_chunk_title(self, chunk):
        """
        If you add a new proposition to a chunk, you may want to update the title or else it can get stale
        """
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a Legal Documentation Expert. A new proposition was just added to a chunk of text from a Supreme Court case.
                    Update the brief chunk title to accurately reflect the legal theme of the propositions.

                    The title must be highly professional, brief (2-5 words), and suitable for a legal index or table of contents.

                    Examples of good titles:
                    - Factual Background: Property Dispute
                    - Appellant's Core Argument
                    - Application of Muluki Ain
                    - Principle of Natural Justice
                    - Final Court Verdict

                    Only respond with the updated chunk title, nothing else. Do not use quotes.
                    """,
                ),
                ("user", "Chunk's propositions:\n{proposition}\n\nChunk summary:\n{current_summary}\n\nCurrent chunk title:\n{current_title}"),
            ]
        )

        runnable = PROMPT | self.llm

        updated_chunk_title = runnable.invoke({
            "proposition": "\n".join(chunk['propositions']),
            "current_summary" : chunk['summary'],
            "current_title" : chunk['title']
        }).content

        return updated_chunk_title

    def _get_new_chunk_summary(self, proposition):
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a Legal Documentation Expert. A new proposition from a Supreme Court case has been identified as a completely new legal topic.
                    Generate a precise, 1-sentence summary that defines what this new chunk will be about.

                    Focus on the legal nature of the proposition. Is it a statement of fact? A citation of a statute? The judge's reasoning?

                    Example:
                    Input: Proposition: The Supreme Court observed that the mother-in-law had already separated after taking her share.
                    Output: This chunk contains the Court's observations and reasoning regarding the separation status of the family members.

                    Only respond with the new chunk summary, nothing else.
                    """,
                ),
                ("user", "Determine the summary of the new chunk that this legal proposition will go into:\n{proposition}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_summary = runnable.invoke({
            "proposition": proposition
        }).content

        return new_chunk_summary

    def _get_new_chunk_title(self, summary):
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a Legal Documentation Expert. Generate a professional, 2-5 word title for a new chunk of Supreme Court text based on its summary.

                    The title should sound like a formal legal heading.

                    Example:
                    Input: Summary: This chunk details the legal boundaries of a guarantor's financial liability under the loan agreement.
                    Output: Guarantor Liability Scope

                    Only respond with the new chunk title, nothing else. Do not use quotes.
                    """,
                ),
                ("user", "Determine the formal legal title of the chunk that this summary belongs to:\n{summary}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_title = runnable.invoke({
            "summary": summary
        }).content

        return new_chunk_title


    def _create_new_chunk(self, proposition):
        new_chunk_id = str(uuid.uuid4())[:self.id_truncate_limit] # I don't want long ids
        new_chunk_summary = self._get_new_chunk_summary(proposition)
        new_chunk_title = self._get_new_chunk_title(new_chunk_summary)

        self.chunks[new_chunk_id] = {
            'chunk_id' : new_chunk_id,
            'propositions': [proposition],
            'title' : new_chunk_title,
            'summary': new_chunk_summary,
            'chunk_index' : len(self.chunks)
        }
        if self.print_logging:
            print (f"Created new chunk ({new_chunk_id}): {new_chunk_title}")

    def get_chunk_outline(self):
        """
        Get a string which represents the chunks you currently have.
        This will be empty when you first start off
        """
        chunk_outline = ""

        for chunk_id, chunk in self.chunks.items():
            single_chunk_string = f"""Chunk ID: {chunk['chunk_id']}\nChunk Name: {chunk['title']}\nChunk Summary: {chunk['summary']}\n\n"""

            chunk_outline += single_chunk_string

        return chunk_outline

    def _find_relevant_chunk(self, proposition):
        current_chunk_outline = self.get_chunk_outline()

        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are a highly skilled Legal Analyst organizing propositions from Nepali Supreme Court case documents.
                    Determine whether the following "Proposition" belongs to any of the existing chunks.

                    A proposition should belong to an existing chunk if it shares the same legal context, such as:
                    - Factual Background (events leading to the case)
                    - Legal Issues & Claims (what the plaintiff/defendant argues)
                    - Statutory Law & Precedents (specific acts, sections, or previous cases cited)
                    - Judicial Reasoning (the court's analysis and logic)
                    - Final Verdict & Principles (the final decision and established legal rules)

                    If you think a proposition logically extends or supports the legal theme of an existing chunk, return that chunk's ID.
                    If the proposition introduces a distinctly new legal argument, a different piece of evidence, or shifts from Facts to Court Reasoning, return "No chunks".

                    Example:
                    Input:
                        - Proposition: "The Finance Company failed to notify the guarantor before auctioning Kitta No. 37."
                        - Current Chunks:
                            - Chunk ID: 2n4l3
                            - Chunk Name: Loan Details & Default
                            - Chunk Summary: Facts regarding the initial loan taken and the subsequent default by the principal debtor.
                            - Chunk ID: 93833
                            - Chunk Name: Violation of Due Process
                            - Chunk Summary: Analysis of how the auction was conducted without proper notice or valuation.
                    Output: 93833
                    """,
                ),
                ("user", "Current Chunks:\n--Start of current chunks--\n{current_chunk_outline}\n--End of current chunks--"),
                ("user", "Determine if the following legal statement should belong to one of the chunks outlined:\n{proposition}"),
            ]
        )

        runnable = PROMPT | self.llm

        chunk_found = runnable.invoke({
            "proposition": proposition,
            "current_chunk_outline": current_chunk_outline
        }).content

        # Pydantic data class
        class ChunkID(BaseModel):
            """Extracting the chunk id"""
            chunk_id: Optional[str]

        # Extraction to catch-all LLM responses. This is a bandaid
        extraction_chain = create_extraction_chain_pydantic(pydantic_schema=ChunkID, llm=self.llm)
        extraction_found = extraction_chain.run(chunk_found)
        if extraction_found:
            chunk_found = extraction_found[0].chunk_id

        # If you got a response that isn't the chunk id limit, chances are it's a bad response or it found nothing
        # So return nothing
        if len(chunk_found) != self.id_truncate_limit:
            return None

        return chunk_found

    def get_chunks(self, get_type='dict'):
        """
        This function returns the chunks in the format specified by the 'get_type' parameter.
        If 'get_type' is 'dict', it returns the chunks as a dictionary.
        If 'get_type' is 'list_of_strings', it returns the chunks as a list of strings, where each string is a proposition in the chunk.
        """
        if get_type == 'dict':
            return self.chunks
        if get_type == 'list_of_strings':
            chunks = []
            for chunk_id, chunk in self.chunks.items():
                chunks.append(" ".join([x for x in chunk['propositions']]))
            return chunks

    def pretty_print_chunks(self):
        print (f"\nYou have {len(self.chunks)} chunks\n")
        for chunk_id, chunk in self.chunks.items():
            print(f"Chunk #{chunk['chunk_index']}")
            print(f"Chunk ID: {chunk_id}")
            print(f"Summary: {chunk['summary']}")
            print(f"Propositions:")
            for prop in chunk['propositions']:
                print(f"    -{prop}")
            print("\n\n")

    def pretty_print_chunk_outline(self):
        print ("Chunk Outline\n")
        print(self.get_chunk_outline())



In [ ]:
ac=AgenticChunker()
ac.add_propositions(eassy_paragraphs)
chunks=ac.get_chunks(get_type='list_of_strings')

